# Cat Behaviour Track Labeller

Label each track by identifying behavior transitions. Dataset will be used to create video classification model.

## Configure notebook

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
import asyncio
import base64
import json
from dataclasses import dataclass

import cv2
import ipywidgets as widgets
import joblib
import numpy as np
from IPython.display import display
from ipyevents import Event

from tracking import _embedding_model_sha256

In [ ]:
# Define constants
DISPLAY_WIDTH = 840
PLAY_INTERVAL_MS = 140
BEHAVIOURS = [
    "walking on floor",
    "walking on surface",
    "drinking",
    "eating from their bowl",
    "eating from wrong bowl",
    "stationary",
    "interacting",
]

In [ ]:
# Define paths
TRACK_ROOT = PROJECT_ROOT / "datasets" / "behaviour_classification_tracks"
ANN_PATH = TRACK_ROOT / "behaviour_labels.json"
TRACK_FILES = sorted(TRACK_ROOT.glob("track-*.joblib"))

## Load and validate tracks

In [ ]:
@dataclass
class TrackInfo:
    track_name: str
    frame_hashes: list[str]
    images: dict
    embedding_model_sha256: str


def load_track(path: Path) -> TrackInfo:
    artifact = joblib.load(path)
    return TrackInfo(
        track_name=path.name,
        frame_hashes=artifact["frame_hashes"],
        images=artifact["images"],
        embedding_model_sha256=artifact["embedding_model_sha256"],
    )


def is_track_annotated(track: TrackInfo, annotations: list[dict]) -> bool:

    # ensure the track was created with the current embedding model
    current_hash = _embedding_model_sha256()
    if track.embedding_model_sha256 != current_hash:
        raise ValueError(f"{track.track_name} used old embeddings model")

    # create hash index map
    hash_map = {h: i for i, h in enumerate(track.frame_hashes) if h}

    # identify annotation windows for the current track
    windows = [
        (hash_map[ann["frame_hash_start"]], hash_map[ann["frame_hash_end"]])
        for ann in annotations
        if ann["track_name"] == track.track_name
    ]

    # evaluate track annotation completeness
    if not windows:
        return False
    elif any(sum(s <= i <= e for s, e in windows) != 1 for i in hash_map.values()):
        raise ValueError(f"{track.track_name} is partially or doubly labelled")
    else:
        return True


annotations = json.loads(ANN_PATH.read_text()) if ANN_PATH.exists() else []
all_tracks = [load_track(path) for path in TRACK_FILES]
pending_tracks = [t for t in all_tracks if not is_track_annotated(t, annotations)]
print(f"Tracks for labelling: {len(pending_tracks)}")
print(f"Complete tracks: {len(all_tracks) - len(pending_tracks)}")

## Define annotation building and saving

In [ ]:
def build_annotation(track: TrackInfo, start: int, end: int, behaviour: str) -> dict:
    if behaviour not in BEHAVIOURS:
        raise ValueError(f"Unknown behavior: {behaviour}")
    if not 0 <= start <= end < len(track.frame_hashes):
        raise ValueError("Annotation boundaries are out of order")
    if not track.frame_hashes[start] or not track.frame_hashes[end]:
        raise ValueError("Annotation boundaries must have frame hashes")
    return {
        "track_name": track.track_name,
        "frame_hash_start": track.frame_hashes[start],
        "frame_hash_end": track.frame_hashes[end],
        "behaviour": behaviour,
    }


def save_annotations(rows: list[dict]) -> None:
    temp_path = ANN_PATH.with_name(f"{ANN_PATH.name}.tmp")
    temp_path.write_text(json.dumps(rows, indent=2) + "\n")
    temp_path.replace(ANN_PATH)

## Define interactive annotation player

In [ ]:
def valid_indices(track: TrackInfo) -> list[int]:
    return [index for index, frame_hash in enumerate(track.frame_hashes) if frame_hash]


def render_track_frame(track: TrackInfo, index: int) -> bytes:
    frame_hash = track.frame_hashes[index]
    image = track.images.get(frame_hash) if frame_hash else None
    if image is None:
        image = np.zeros((360, DISPLAY_WIDTH, 3), dtype=np.uint8)
        cv2.putText(
            image,
            "No ROI image for this frame",
            (24, 180),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (210, 210, 210),
            2,
            cv2.LINE_AA,
        )
    else:
        height, width = image.shape[:2]
        scale = min(DISPLAY_WIDTH / width, 720 / height, 1.0)
        image = cv2.resize(
            image,
            (int(width * scale), int(height * scale)),
            interpolation=cv2.INTER_AREA,
        )

    success, encoded = cv2.imencode(".jpg", image)
    if not success:
        raise RuntimeError("OpenCV could not encode the preview frame")
    return encoded.tobytes()

In [ ]:
track_index = 0
segment_start = None
active_behaviour = None

frame_view = widgets.HTML(
    layout=widgets.Layout(width=f"{DISPLAY_WIDTH}px", border="1px solid #555")
)
frame_slider = widgets.IntSlider(
    description="Frame",
    min=0,
    max=0,
    value=0,
    continuous_update=True,
    layout=widgets.Layout(width="70%"),
)
play_toggle = widgets.ToggleButton(value=False, description="Play", icon="play")
behaviour_control = widgets.Dropdown(
    options=BEHAVIOURS,
    description="Behaviour",
    layout=widgets.Layout(width="360px"),
)
start_button = widgets.Button(description="Start behavior", icon="play")
transition_button = widgets.Button(description="Transition here", icon="exchange")
finish_button = widgets.Button(description="Finish track", icon="check")
keyboard_target = widgets.HTML(
    value=(
        '<div tabindex="0" style="padding:8px;border:1px solid #bbb;max-width:440px">'
        "Click here, then use Space or the arrow keys</div>"
    )
)
keyboard_event = Event(
    source=keyboard_target,
    watched_events=["keydown"],
    prevent_default_action=True,
)
status = widgets.HTML()

In [ ]:
def refresh_view(*_) -> None:
    if not pending_tracks or track_index >= len(pending_tracks):
        status.value = "All tracks have exactly one behavior per frame."
        return

    track = pending_tracks[track_index]
    frame_hash = track.frame_hashes[frame_slider.value]
    status.value = (
        f"<b>{track.track_name}</b> &nbsp; "
        f"frame {frame_slider.value + 1}/{len(track.frame_hashes)}"
        f" &nbsp; hash: {frame_hash}"
        f" &nbsp; selected: {behaviour_control.value}"
        f" &nbsp; active: {active_behaviour or 'not started'}"
    )
    encoded = base64.b64encode(render_track_frame(track, frame_slider.value)).decode(
        "ascii"
    )
    frame_view.value = (
        f'<img src="data:image/jpeg;base64,{encoded}" '
        'style="display:block;max-width:100%;height:auto">'
    )


def set_frame(index: int) -> None:
    frame_slider.value = max(frame_slider.min, min(index, frame_slider.max))


def current_valid_index() -> int | None:
    return next(
        (
            index
            for index in valid_indices(pending_tracks[track_index])
            if index >= frame_slider.value
        ),
        None,
    )


def load_next_track() -> None:
    if track_index >= len(pending_tracks):
        frame_view.value = ""
        status.value = "All tracks have exactly one behavior per frame."
        return

    track = pending_tracks[track_index]
    indices = valid_indices(track)
    if not indices:
        frame_view.value = ""
        status.value = f"{track.track_name} has no frames with hashes."
        return

    frame_slider.max = len(track.frame_hashes) - 1
    frame_slider.value = indices[0]
    play_toggle.value = False
    refresh_view()

In [ ]:
def start_behaviour(*_) -> None:
    global segment_start, active_behaviour

    start = current_valid_index()
    if start is None:
        status.value = "No hashed frame is available at this position."
        return

    segment_start = start
    active_behaviour = behaviour_control.value
    refresh_view()


def save_transition(*_) -> None:
    global segment_start, active_behaviour

    if segment_start is None:
        status.value = "Choose the initial behavior and click Start behavior."
        return

    track = pending_tracks[track_index]
    transition = current_valid_index()
    if transition is None or transition <= segment_start:
        status.value = "Move forward to the first frame of the new behavior."
        return

    end = max(index for index in valid_indices(track) if index < transition)
    annotations.append(build_annotation(track, segment_start, end, active_behaviour))
    save_annotations(annotations)
    segment_start, active_behaviour = transition, behaviour_control.value
    refresh_view()


def finish_track(*_) -> None:
    global track_index, segment_start, active_behaviour

    if segment_start is None:
        status.value = "Choose the initial behavior and click Start behavior."
        return

    track = pending_tracks[track_index]
    annotations.append(
        build_annotation(
            track, segment_start, valid_indices(track)[-1], active_behaviour
        )
    )
    save_annotations(annotations)
    track_index += 1
    segment_start = active_behaviour = None
    load_next_track()

In [ ]:
def handle_key(event: dict) -> None:
    key = event.get("key", "").lower()
    if key in (" ", "space"):
        play_toggle.value = not play_toggle.value
    elif key in ("arrowleft", "left"):
        set_frame(frame_slider.value - 1)
    elif key in ("arrowright", "right"):
        set_frame(frame_slider.value + 1)


def start_labelling() -> None:
    controls = widgets.VBox(
        [
            widgets.HBox([play_toggle, frame_slider]),
            widgets.HBox(
                [behaviour_control, start_button, transition_button, finish_button]
            ),
            keyboard_target,
            status,
            frame_view,
        ]
    )
    display(controls)

    keyboard_event.on_dom_event(handle_key)
    frame_slider.observe(refresh_view, names="value")
    behaviour_control.observe(refresh_view, names="value")
    start_button.on_click(start_behaviour)
    transition_button.on_click(save_transition)
    finish_button.on_click(finish_track)

    async def playback_loop() -> None:
        while play_toggle.value and track_index < len(pending_tracks):
            if frame_slider.value >= frame_slider.max:
                play_toggle.value = False
                break
            frame_slider.value += 1
            await asyncio.sleep(PLAY_INTERVAL_MS / 1000)

    def on_play_toggle(change: dict) -> None:
        play_toggle.icon = "pause" if change["new"] else "play"
        if change["new"]:
            asyncio.create_task(playback_loop())

    play_toggle.observe(on_play_toggle, names="value")
    load_next_track()

## Label tracks

In [ ]:
start_labelling()